# Draft Notebook - Get Fanwork Links with troubleshooting
This notebook contains functions that:
1) Parses a list of pagination links for works associated with any given Ao3 fandom (pagination link example: 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3').
2) For each pagination link, scrapes that link for a list of links to fanworks (fanwork link example: 'https://archiveofourown.org/works/80602681/chapters/211688701'.
3) Compiles all fanwork links into a list.

In [57]:
from draft_get_list_of_links import works_only

all_fanwork_links = []
for filepath in filepath_list:
    soup = read_local_html_file(filepath)
    all_links = get_all_links(soup)
    urls_only = get_urls_only(all_links)
    work_links = get_work_links(urls_only)
    work_links_pared = pare_down_work_links(work_links)
    fanworks_list = remove_chapter_links(work_links_pared)
    all_fanwork_links.extend(fanworks_list)

In [7]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re

# Parse a list of pagination links

In [1]:
def get_links_to_scrape(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    links_to_scrape = open(filepath).readlines()
    return links_to_scrape

In [4]:
# test get_links_to_scrape
links_to_scrape = get_links_to_scrape('../../data/pagination_links.txt')
print(links_to_scrape)

['https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=2\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=4\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=5\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=6\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=7\n', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=8\n', 'https://archiveofourown.org/tags/Cande

# Scrape individual pagination link for list of fanwork links

In [6]:
def get_all_links(soup):
    """Takes in a BeautifulSoup object and returns a list of all links in the html document."""
    links = soup.find_all('a')
    for link in links:
        link = link.get('href')
    return links

In [8]:
## test get_all_links

# set filepath
candela_filepath = '../../data/Candela Obscura (Critical Role Web Series) - Works _ Archive of Our Own.html'

# create BeautifulSoup object
with open(candela_filepath, 'r', encoding='utf-8') as file:
    html_content = file.read()
    candela_soup = BeautifulSoup(html_content, 'html.parser')

# call get_all_links
all_links = get_all_links(candela_soup)
print(all_links)

[<a href="https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?page=1#main">Main Content</a>, <a href="https://archiveofourown.org/"><span>Archive of Our Own</span><img alt="Archive of Our Own" class="logo" src="./Candela Obscura (Critical Role Web Series) - Works _ Archive of Our Own_files/logo_42.png"/></a>, <a class="dropdown-toggle" data-target="#" data-toggle="dropdown" href="https://archiveofourown.org/users/OceanTiger23">Hi, OceanTiger23!</a>, <a href="https://archiveofourown.org/users/OceanTiger23">My Dashboard</a>, <a href="https://archiveofourown.org/users/OceanTiger23/subscriptions">My Subscriptions</a>, <a href="https://archiveofourown.org/users/OceanTiger23/works">My Works</a>, <a href="https://archiveofourown.org/users/OceanTiger23/bookmarks">My Bookmarks</a>, <a href="https://archiveofourown.org/users/OceanTiger23/readings">My History</a>, <a href="https://archiveofourown.org/users/OceanTiger23/preferences">My Preferences</a>, <a cl

In [12]:
def get_urls_only(links):
    """Parses a list of links from a BeautifulSoup object and returns only urls without associated html code."""
    # use regex to get just URLs
    urls_only = []
    pattern = r'\"(.*?)\"'
    for i in range(len(links)):
        link_str = str(links[i])
        match = re.search(pattern, link_str)
        if match:
            urls_only.append(match.group(1))
        else:
            continue

    return urls_only

In [19]:
# test get_urls_only
urls_only = get_urls_only(all_links)
print(type(urls_only[0]))
print("There are", len(urls_only), "URLs on this page.")
print(urls_only)

<class 'str'>
There are 699 URLs on this page.
['https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?page=1#main', 'https://archiveofourown.org/', 'dropdown-toggle', 'https://archiveofourown.org/users/OceanTiger23', 'https://archiveofourown.org/users/OceanTiger23/subscriptions', 'https://archiveofourown.org/users/OceanTiger23/works', 'https://archiveofourown.org/users/OceanTiger23/bookmarks', 'https://archiveofourown.org/users/OceanTiger23/readings', 'https://archiveofourown.org/users/OceanTiger23/preferences', 'dropdown-toggle', 'https://archiveofourown.org/works/new', 'https://archiveofourown.org/works/new?import=true', 'delete', 'https://archiveofourown.org/users/OceanTiger23', 'dropdown-toggle', 'https://archiveofourown.org/media', 'https://archiveofourown.org/media/Anime%20*a*%20Manga/fandoms', 'https://archiveofourown.org/media/Books%20*a*%20Literature/fandoms', 'https://archiveofourown.org/media/Cartoons%20*a*%20Comics%20*a*%20Graphic%20No

In [103]:
def update_urls_only(urls_only):
    for i in range(len(urls_only)):
        if urls_only[i][0] == '/':
            urls_only[i] = 'https://archiveofourown.org' + urls_only[i]
        else:
            continue
    return urls_only

In [23]:
def get_work_links(urls_only):
    """Pares down a list of URLs from get_urls_only to a list of URLs associated with fanworks (as opposd to Ao3 tags, bookmarks, collections, search, users, etc."""
    work_links = []
    for i in range(len(urls_only)):
        if 'https://archiveofourown.org/works/' in urls_only[i]:
            work_links.append(urls_only[i])
        else:
            continue
    return work_links

In [24]:
# test get_work_links
work_links = get_work_links(urls_only)
print(work_links)

['https://archiveofourown.org/works/new', 'https://archiveofourown.org/works/new?import=true', 'https://archiveofourown.org/works/search', 'https://archiveofourown.org/works/71539056', 'https://archiveofourown.org/works/71539056/chapters/226429976', 'https://archiveofourown.org/works/71539056/collections', 'https://archiveofourown.org/works/71539056?show_comments=true&amp;view_full_work=true#comments', 'https://archiveofourown.org/works/71539056?view_full_work=true#kudos', 'https://archiveofourown.org/works/71539056/bookmarks', 'https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/80602681/chapters/223632446', 'https://archiveofourown.org/works/80602681?show_comments=true&amp;view_full_work=true#comments', 'https://archiveofourown.org/works/80602681?view_full_work=true#kudos', 'https://archiveofourown.org/works/80602681/bookmarks', 'https://archiveofourown.org/works/84168161', 'https://archiveofourown.org/works/84168161?show_comments=true#comments', 'https://

In [30]:
def pare_down_work_links(work_links):
    """Takes a list of fanwork links from get_work_links and pares down to links that only end in digits (i.e. the link to the first page of an individual fanwork, rather than links to kudos, comments, chapters, etc. associated with that fanwork)."""
    works_only = []
    pattern = r'\d+$'
    for i in range(len(work_links)):
        match = re.search(pattern, work_links[i])
        if match:
            works_only.append(work_links[i])
        else:
            continue

    return works_only

In [33]:
# test pare_down_work_links
work_links1 = pare_down_work_links(work_links)
print(work_links1)

['https://archiveofourown.org/works/71539056', 'https://archiveofourown.org/works/71539056/chapters/226429976', 'https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/80602681/chapters/223632446', 'https://archiveofourown.org/works/84168161', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51518734/chapters/220941681', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/51010177/chapters/212450026', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/74479381/chapters/203838821', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/56574286/chapters/202821756', 'https://archiveofourown.org/works/75883311', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/56635081', 'https://archiveofourown.org/works/56635081/chapters/197756806', 'https://archiveofourown.org/works/739

In [164]:
def remove_chapter_links(work_links_pared):
    """Takes in a list of pared down fanwork URLs from pare_down_work_links and removes any links to individual fanwork chapters."""
    works_only_final = work_links_pared
    for i in range(len(work_links_pared)):
        current_link = work_links_pared[i]
        if 'chapters' in current_link:
            works_only_final.remove(current_link)
        else:
            continue
    return works_only_final

In [155]:
def remove_chapter_links_test(work_links_pared):
    """Takes in a list of pared down fanwork URLs from pare_down_work_links and removes any links to individual fanwork chapters."""
    works_only_final = work_links_pared
    for i in range(len(work_links_pared)):
        current_link = work_links_pared[i]
        if 'chapters' in current_link:
            works_only_final.remove(current_link)
        else:
            continue
    return works_only_final

In [39]:
# test remove_chapter_links
fanworks_list = remove_chapter_links(work_links1)
print("This is the final list of fanwork links to scrape.")
print("There are", len(fanworks_list), "links to scrape.")
print(fanworks_list)

This is the final list of fanwork links to scrape.
There are 20 links to scrape.
['https://archiveofourown.org/works/71539056', 'https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/84168161', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/75883311', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/56635081', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/72821951', 'https://archiveofourown.org/works/63168412', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606', 'https://archiveofourown.org/works/69266581', 'https://archiveofourown.org/works/68678736', 'https://archiveofourown.org/works/59798779', 'https://archiveofourown.org/works/55118308'

Now that I know this process works, I can get fanwork links from all the pagination links in "links_to_scrape."

# Test - save beautifulsoup object to file

In [127]:
def get_html_doc(url):
    """Takes in the url for the first page of works for any given Ao3 fandom and returns an html document of that page."""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

# pretty sure get_html_doc needs to have response.text, not response.content

In [43]:
for i in range(len(links_to_scrape)):
    soup = get_html_doc(links_to_scrape[i])
    html = soup.prettify("utf-8")
    page_number = i
    filepath = 'data/page_' + str(i) + '.html'
    with open(filepath, "wb") as file:
        file.write(html)

# code adapted from: https://stackoverflow.com/questions/40529848/how-to-write-the-output-to-html-file-with-python-beautifulsoup

In [51]:
file_list = []
for i in range(len(links_to_scrape)):
    filepath = 'page_' + str(i) + '.html'
    file_list.append(filepath)
print(file_list)

['page_0.html', 'page_1.html', 'page_2.html', 'page_3.html', 'page_4.html', 'page_5.html', 'page_6.html', 'page_7.html', 'page_8.html', 'page_9.html', 'page_10.html']


# Compile all fanwork links into a list

In [81]:
def read_local_html_file(filepath):
    """Takes in a filepath to a local html file and returns a BeautifulSoup object."""
    with open(filepath, 'r', encoding='utf-8') as file:
        html_content = file.read()
        soup = BeautifulSoup(html_content, 'html.parser')
    return soup

In [59]:
soup1 = read_local_html_file('../../data/page_1.html')

In [61]:
all_links1 = get_all_links(soup1)
print(all_links1)

# get_all_links is not returning full links with 'http://archiveofourown.org/' at the front...

[<a href="#main">
      Main Content
     </a>, <a href="/">
<span>
       Archive of Our Own
      </span>
<img alt="Archive of Our Own" class="logo" src="/images/ao3_logos/logo_42.png"/>
</a>, <a href="/users/login?return_to=%2Ftags%2FCandela%2520Obscura%2520%28Critical%2520Role%2520Web%2520Series%29%2Fworks%3Fview_adult%3Dtrue%26view_adult%3Dtrue%26page%3D2%250A" id="login-dropdown">
       Log In
      </a>, <a href="/users/password/new">
         Forgot password?
        </a>, <a href="/invite_requests">
         Get an Invitation
        </a>, <a href="/menu/fandoms">
        Fandoms
       </a>, <a href="/media">
          All Fandoms
         </a>, <a href="/media/Anime%20*a*%20Manga/fandoms">
          Anime &amp; Manga
         </a>, <a href="/media/Books%20*a*%20Literature/fandoms">
          Books &amp; Literature
         </a>, <a href="/media/Cartoons%20*a*%20Comics%20*a*%20Graphic%20Novels/fandoms">
          Cartoons &amp; Comics &amp; Graphic Novels
         </a>, <a h

In [62]:
urls_only1 = get_urls_only(all_links1)
print(urls_only1)

['#main', '/', '/users/login?return_to=%2Ftags%2FCandela%2520Obscura%2520%28Critical%2520Role%2520Web%2520Series%29%2Fworks%3Fview_adult%3Dtrue%26view_adult%3Dtrue%26page%3D2%250A', '/users/password/new', '/invite_requests', '/menu/fandoms', '/media', '/media/Anime%20*a*%20Manga/fandoms', '/media/Books%20*a*%20Literature/fandoms', '/media/Cartoons%20*a*%20Comics%20*a*%20Graphic%20Novels/fandoms', '/media/Celebrities%20*a*%20Real%20People/fandoms', '/media/Movies/fandoms', '/media/Music%20*a*%20Bands/fandoms', '/media/Other%20Media/fandoms', '/media/Theater/fandoms', '/media/TV%20Shows/fandoms', '/media/Video%20Games/fandoms', '/media/Uncategorized%20Fandoms/fandoms', '/menu/browse', '/works', '/bookmarks', '/tags', '/collections', '/menu/search', '/works/search', '/bookmarks/search', '/tags/search', '/people/search', '/menu/about', '/about', '/admin_posts', '/faq', '/wrangling_guidelines', '/donate', 'tag', '/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/bookmarks', '#work-

In [63]:
works_links1 = get_work_links(urls_only1)
print(works_links1)

[]


In [58]:
print(len(all_fanwork_links))

0


In [ ]:
def update_urls_only(urls_only):
    for i in range(len(urls_only)):
        if urls_only[i][0] == '/':
            urls_only[i] = 'https://archiveofourown.org' + urls_only[i]
        else:
            continue
    return urls_only

In [66]:
print(len(all_fanwork_links1))

0


In [67]:
for i in range(len(links_to_scrape)):
    page = requests.get(links_to_scrape[i])
    filepath = 'data/page' + str(i+1) + '.html'
    with open(filepath, "wb+") as f:
        f.write(page.content)

In [71]:
filepath_list = []
for i in range(len(links_to_scrape)):
    filepath = 'data/page_' + str(i+1) + '.html'
    filepath_list.append(filepath)

In [72]:
print(filepath_list)

['data/page_1.html', 'data/page_2.html', 'data/page_3.html', 'data/page_4.html', 'data/page_5.html', 'data/page_6.html', 'data/page_7.html', 'data/page_8.html', 'data/page_9.html', 'data/page_10.html', 'data/page_11.html']


In [73]:
all_fanwork_links = []
for filepath in filepath_list:
    soup = read_local_html_file(filepath)
    all_links = get_all_links(soup)
    urls_only = get_urls_only(all_links)
    work_links = get_work_links(urls_only)
    work_links_pared = pare_down_work_links(work_links)
    fanworks_list = remove_chapter_links(work_links_pared)
    all_fanwork_links.extend(fanworks_list)

In [74]:
print(all_fanwork_links)

[]


# Troubleshooting

In [104]:
test_file = '../../data/page_3.html'

In [82]:
all_fanworks_links = []

In [89]:
print(links_to_scrape[2])

https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3



In [97]:
soup3 = get_html_doc('https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3')

In [98]:
print(soup3)

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="ie=edge" http-equiv="x-ua-compatible"/>
<meta content="fanfiction, transformative works, otw, fair use, archive" name="keywords"/>
<meta content="en-US" name="language"/>
<meta content="fandom" name="subject"/>
<meta content="An Archive of Our Own, a project of the Organization for Transformative Works" name="description"/>
<meta content="GLOBAL" name="distribution"/>
<meta content="transformative works" name="classification"/>
<meta content="Organization for Transformative Works" name="author"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<meta content="nointentdetection" name="chrome"/>
<meta content="telephone=no" name="format-detection"/>
<title>Candela Obscura (Critical Role Web Series) - Works | Archive of Our Own</title>
<link href="/stylesheets/skins/skin_1_default/1_site_screen_.css" media="screen" rel="stylesheet" type="text/css"/>
<link href="/stylesheets/skins/skin_1_

In [105]:
all_links3 = get_all_links(soup3)
print(all_links3)

# get_all_links works

[<a href="#main">Main Content</a>, <a href="/"><span>Archive of Our Own</span><img alt="Archive of Our Own" class="logo" src="/images/ao3_logos/logo_42.png"/></a>, <a href="/users/login?return_to=%2Ftags%2FCandela%2520Obscura%2520%28Critical%2520Role%2520Web%2520Series%29%2Fworks%3Fview_adult%3Dtrue%26view_adult%3Dtrue%26page%3D3" id="login-dropdown">Log In</a>, <a href="/users/password/new">Forgot password?</a>, <a href="/invite_requests">Get an Invitation</a>, <a href="/menu/fandoms">Fandoms</a>, <a href="/media">All Fandoms</a>, <a href="/media/Anime%20*a*%20Manga/fandoms">Anime &amp; Manga</a>, <a href="/media/Books%20*a*%20Literature/fandoms">Books &amp; Literature</a>, <a href="/media/Cartoons%20*a*%20Comics%20*a*%20Graphic%20Novels/fandoms">Cartoons &amp; Comics &amp; Graphic Novels</a>, <a href="/media/Celebrities%20*a*%20Real%20People/fandoms">Celebrities &amp; Real People</a>, <a href="/media/Movies/fandoms">Movies</a>, <a href="/media/Music%20*a*%20Bands/fandoms">Music &amp;

In [106]:
urls_only3 = get_urls_only(all_links3)
print(urls_only3)

# urls only works

['#main', '/', '/users/login?return_to=%2Ftags%2FCandela%2520Obscura%2520%28Critical%2520Role%2520Web%2520Series%29%2Fworks%3Fview_adult%3Dtrue%26view_adult%3Dtrue%26page%3D3', '/users/password/new', '/invite_requests', '/menu/fandoms', '/media', '/media/Anime%20*a*%20Manga/fandoms', '/media/Books%20*a*%20Literature/fandoms', '/media/Cartoons%20*a*%20Comics%20*a*%20Graphic%20Novels/fandoms', '/media/Celebrities%20*a*%20Real%20People/fandoms', '/media/Movies/fandoms', '/media/Music%20*a*%20Bands/fandoms', '/media/Other%20Media/fandoms', '/media/Theater/fandoms', '/media/TV%20Shows/fandoms', '/media/Video%20Games/fandoms', '/media/Uncategorized%20Fandoms/fandoms', '/menu/browse', '/works', '/bookmarks', '/tags', '/collections', '/menu/search', '/works/search', '/bookmarks/search', '/tags/search', '/people/search', '/menu/about', '/about', '/admin_posts', '/faq', '/wrangling_guidelines', '/donate', 'tag', '/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/bookmarks', '#work-filte

In [107]:
urls_only3_updated = update_urls_only(urls_only3)
print(urls_only3_updated)

['#main', 'https://archiveofourown.org/', 'https://archiveofourown.org/users/login?return_to=%2Ftags%2FCandela%2520Obscura%2520%28Critical%2520Role%2520Web%2520Series%29%2Fworks%3Fview_adult%3Dtrue%26view_adult%3Dtrue%26page%3D3', 'https://archiveofourown.org/users/password/new', 'https://archiveofourown.org/invite_requests', 'https://archiveofourown.org/menu/fandoms', 'https://archiveofourown.org/media', 'https://archiveofourown.org/media/Anime%20*a*%20Manga/fandoms', 'https://archiveofourown.org/media/Books%20*a*%20Literature/fandoms', 'https://archiveofourown.org/media/Cartoons%20*a*%20Comics%20*a*%20Graphic%20Novels/fandoms', 'https://archiveofourown.org/media/Celebrities%20*a*%20Real%20People/fandoms', 'https://archiveofourown.org/media/Movies/fandoms', 'https://archiveofourown.org/media/Music%20*a*%20Bands/fandoms', 'https://archiveofourown.org/media/Other%20Media/fandoms', 'https://archiveofourown.org/media/Theater/fandoms', 'https://archiveofourown.org/media/TV%20Shows/fandoms'

In [108]:
work_links3 = get_work_links(urls_only3_updated)
print(work_links3)

# works now after adding update_urls_only function

['https://archiveofourown.org/works/search', 'https://archiveofourown.org/works/56112163', 'https://archiveofourown.org/works/56112163?show_comments=true#comments', 'https://archiveofourown.org/works/56112163#kudos', 'https://archiveofourown.org/works/56112163/bookmarks', 'https://archiveofourown.org/works/56104648', 'https://archiveofourown.org/works/56104648#kudos', 'https://archiveofourown.org/works/56104648/bookmarks', 'https://archiveofourown.org/works/55897618', 'https://archiveofourown.org/works/55897618?show_comments=true#comments', 'https://archiveofourown.org/works/55897618#kudos', 'https://archiveofourown.org/works/55897618/bookmarks', 'https://archiveofourown.org/works/55858051', 'https://archiveofourown.org/works/55849537', 'https://archiveofourown.org/works/55849537?show_comments=true#comments', 'https://archiveofourown.org/works/55849537#kudos', 'https://archiveofourown.org/works/55849537/bookmarks', 'https://archiveofourown.org/works/55829683', 'https://archiveofourown.

In [109]:
work_links_pared3 = pare_down_work_links(work_links3)
print(work_links_pared3)

['https://archiveofourown.org/works/56112163', 'https://archiveofourown.org/works/56104648', 'https://archiveofourown.org/works/55897618', 'https://archiveofourown.org/works/55858051', 'https://archiveofourown.org/works/55849537', 'https://archiveofourown.org/works/55829683', 'https://archiveofourown.org/works/55821961', 'https://archiveofourown.org/works/55602580', 'https://archiveofourown.org/works/55469746', 'https://archiveofourown.org/works/55469746/chapters/141333463', 'https://archiveofourown.org/works/55495195', 'https://archiveofourown.org/works/55447729', 'https://archiveofourown.org/works/55024471', 'https://archiveofourown.org/works/55024471/chapters/140673469', 'https://archiveofourown.org/works/54959626', 'https://archiveofourown.org/works/54959626/chapters/140671291', 'https://archiveofourown.org/works/55205539', 'https://archiveofourown.org/works/55188448', 'https://archiveofourown.org/works/55022362', 'https://archiveofourown.org/works/54638023', 'https://archiveofouro

In [112]:
print(type(work_links_pared3[0]))

<class 'str'>


In [113]:
works_only_final3 = work_links_pared3

In [117]:
fanworks_list3 = remove_chapter_links(work_links_pared3)
print(fanworks_list3)

# throwing an error

['https://archiveofourown.org/works/56112163', 'https://archiveofourown.org/works/56104648', 'https://archiveofourown.org/works/55897618', 'https://archiveofourown.org/works/55858051', 'https://archiveofourown.org/works/55849537', 'https://archiveofourown.org/works/55829683', 'https://archiveofourown.org/works/55821961', 'https://archiveofourown.org/works/55602580', 'https://archiveofourown.org/works/55469746', 'https://archiveofourown.org/works/55495195', 'https://archiveofourown.org/works/55447729', 'https://archiveofourown.org/works/55024471', 'https://archiveofourown.org/works/54959626', 'https://archiveofourown.org/works/55205539', 'https://archiveofourown.org/works/55188448', 'https://archiveofourown.org/works/55022362', 'https://archiveofourown.org/works/54638023', 'https://archiveofourown.org/works/54777958', 'https://archiveofourown.org/works/54682489', 'https://archiveofourown.org/works/53537782']


In [118]:
print(len(fanworks_list3))

20


In [119]:
all_fanwork_links_test = []

In [120]:
all_fanwork_links_test.extend(fanworks_list3)

In [121]:
print(all_fanwork_links_test)

['https://archiveofourown.org/works/56112163', 'https://archiveofourown.org/works/56104648', 'https://archiveofourown.org/works/55897618', 'https://archiveofourown.org/works/55858051', 'https://archiveofourown.org/works/55849537', 'https://archiveofourown.org/works/55829683', 'https://archiveofourown.org/works/55821961', 'https://archiveofourown.org/works/55602580', 'https://archiveofourown.org/works/55469746', 'https://archiveofourown.org/works/55495195', 'https://archiveofourown.org/works/55447729', 'https://archiveofourown.org/works/55024471', 'https://archiveofourown.org/works/54959626', 'https://archiveofourown.org/works/55205539', 'https://archiveofourown.org/works/55188448', 'https://archiveofourown.org/works/55022362', 'https://archiveofourown.org/works/54638023', 'https://archiveofourown.org/works/54777958', 'https://archiveofourown.org/works/54682489', 'https://archiveofourown.org/works/53537782']


# Testing again post-troubleshooting

In [147]:
all_fanwork_links2 = []

for i in range(len(links_to_scrape)):
    print(links_to_scrape[i])
    soup = get_html_doc(links_to_scrape[i])
    print("Soup successfully created for", str(links_to_scrape[i]))
    all_links = get_all_links(soup)
    print("All links list successfully created for", str(links_to_scrape[i]))
    urls_only = get_urls_only(all_links)
    print("URLS only list successfully created for", str(links_to_scrape[i]))
    urls_only_updated = update_urls_only(urls_only)
    print("URLS only list successfully updated for", str(links_to_scrape[i]))
    work_links = get_work_links(urls_only_updated)
    print("Work links list successfully created for", str(links_to_scrape[i]))
    work_links_pared = pare_down_work_links(work_links)
    print("Work links list successfully pared down for", str(links_to_scrape[i]))
    all_fanwork_links2.extend(work_links_pared)
    print("List of work links for", str(links_to_scrape[i]), "successfully added to all_fanwork_links2.")

https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1

Soup successfully created for https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1

All links list successfully created for https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1

URLS only list successfully created for https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1

URLS only list successfully updated for https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1

Work links list successfully created for https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1

Work links list successfully pared down for https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role

In [148]:
print(all_fanwork_links2)

['https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/80602681/chapters/223632446', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51518734/chapters/220941681', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/51010177/chapters/212450026', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/74479381/chapters/203838821', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/56574286/chapters/202821756', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606', 'https://archiveofourown.org/works/69266581', 'https://archiveofourown.org/works/68678736', 'https://archiveofourown.org/works/59798779', 'https://archiveofourown.org/works/59798779/chapters/174878791', 'https://ar

In [149]:
print(len(all_fanwork_links2))

201


In [151]:
all_fanwork_links2_updated = all_fanwork_links2
for i in range(len(all_fanwork_links2)):
    if 'chapter' in all_fanwork_links2[i]:
        all_fanwork_links2_updated.remove(all_fanwork_links2[i])
    else:
        continue

In [152]:
print(all_fanwork_links2_updated)

['https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606', 'https://archiveofourown.org/works/69266581', 'https://archiveofourown.org/works/68678736', 'https://archiveofourown.org/works/59798779', 'https://archiveofourown.org/works/55118308', 'https://archiveofourown.org/works/66001534', 'https://archiveofourown.org/works/62477026', 'https://archiveofourown.org/works/62036794', 'https://archiveofourown.org/works/60847894', 'https://archiveofourown.org/works/51300544', 'https://archiveofourown.org/works/56817901', 'https://archiveofourown.org/works/56876515', 'https://archiveofourown.org/work

In [153]:
print(len(all_fanwork_links2_updated))

171


In [165]:
# making the chapter thing separate 'cause I have no idea what the deal is
all_fanwork_links2_updated = remove_chapter_links(all_fanwork_links2)
print(all_fanwork_links2_updated)

['https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606', 'https://archiveofourown.org/works/69266581', 'https://archiveofourown.org/works/68678736', 'https://archiveofourown.org/works/59798779', 'https://archiveofourown.org/works/55118308', 'https://archiveofourown.org/works/66001534', 'https://archiveofourown.org/works/62477026', 'https://archiveofourown.org/works/62036794', 'https://archiveofourown.org/works/60847894', 'https://archiveofourown.org/works/51300544', 'https://archiveofourown.org/works/56817901', 'https://archiveofourown.org/works/56876515', 'https://archiveofourown.org/work

# Troubleshooting round 2

In [130]:
test_url = str(links_to_scrape[3])
print(test_url)

https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=4



In [131]:
all_fanwork_links4 = []

In [133]:
soup4 = get_html_doc(test_url)
all_links4 = get_all_links(soup4)
urls_only4 = get_urls_only(all_links4)
urls_only_updated4 = update_urls_only(urls_only4)
work_links4 = get_work_links(urls_only_updated4)
work_links_pared4 = pare_down_work_links(work_links4)

In [134]:
print(work_links_pared4)

['https://archiveofourown.org/works/54270886', 'https://archiveofourown.org/works/54199738', 'https://archiveofourown.org/works/54066016', 'https://archiveofourown.org/works/54066016/chapters/136993645', 'https://archiveofourown.org/works/54070948', 'https://archiveofourown.org/works/54070948/chapters/143768602', 'https://archiveofourown.org/works/53756806', 'https://archiveofourown.org/works/53629054', 'https://archiveofourown.org/works/53629054/chapters/136001023', 'https://archiveofourown.org/works/53636989', 'https://archiveofourown.org/works/53557639', 'https://archiveofourown.org/works/53531353', 'https://archiveofourown.org/works/53501956', 'https://archiveofourown.org/works/53498773', 'https://archiveofourown.org/works/53485774', 'https://archiveofourown.org/works/53410531', 'https://archiveofourown.org/works/53358775', 'https://archiveofourown.org/works/53275300', 'https://archiveofourown.org/works/53228317', 'https://archiveofourown.org/works/53141194', 'https://archiveofouro

In [136]:
print(type(work_links_pared4))

<class 'list'>


In [166]:
fanworks_list5 = remove_chapter_links(work_links_pared4)
print(fanworks_list5)

['https://archiveofourown.org/works/54270886', 'https://archiveofourown.org/works/54199738', 'https://archiveofourown.org/works/54066016', 'https://archiveofourown.org/works/54070948', 'https://archiveofourown.org/works/53756806', 'https://archiveofourown.org/works/53629054', 'https://archiveofourown.org/works/53636989', 'https://archiveofourown.org/works/53557639', 'https://archiveofourown.org/works/53531353', 'https://archiveofourown.org/works/53501956', 'https://archiveofourown.org/works/53498773', 'https://archiveofourown.org/works/53485774', 'https://archiveofourown.org/works/53410531', 'https://archiveofourown.org/works/53358775', 'https://archiveofourown.org/works/53275300', 'https://archiveofourown.org/works/53228317', 'https://archiveofourown.org/works/53141194', 'https://archiveofourown.org/works/53209828', 'https://archiveofourown.org/works/53208346', 'https://archiveofourown.org/works/53108299']


In [167]:
print(len(fanworks_list5))

20


In [137]:
works_only_final4 = work_links_pared4
print(works_only_final4)

['https://archiveofourown.org/works/54270886', 'https://archiveofourown.org/works/54199738', 'https://archiveofourown.org/works/54066016', 'https://archiveofourown.org/works/54070948', 'https://archiveofourown.org/works/53756806', 'https://archiveofourown.org/works/53629054', 'https://archiveofourown.org/works/53636989', 'https://archiveofourown.org/works/53557639', 'https://archiveofourown.org/works/53531353', 'https://archiveofourown.org/works/53501956', 'https://archiveofourown.org/works/53498773', 'https://archiveofourown.org/works/53485774', 'https://archiveofourown.org/works/53410531', 'https://archiveofourown.org/works/53358775', 'https://archiveofourown.org/works/53275300', 'https://archiveofourown.org/works/53228317', 'https://archiveofourown.org/works/53141194', 'https://archiveofourown.org/works/53209828', 'https://archiveofourown.org/works/53208346', 'https://archiveofourown.org/works/53108299']


In [139]:
for i in range(len(work_links_pared4)):
    if 'chapters' in work_links_pared4[i]:
        works_only_final4.remove(work_links_pared[i])

print(works_only_final4)

['https://archiveofourown.org/works/54270886', 'https://archiveofourown.org/works/54199738', 'https://archiveofourown.org/works/54066016', 'https://archiveofourown.org/works/54070948', 'https://archiveofourown.org/works/53756806', 'https://archiveofourown.org/works/53629054', 'https://archiveofourown.org/works/53636989', 'https://archiveofourown.org/works/53557639', 'https://archiveofourown.org/works/53531353', 'https://archiveofourown.org/works/53501956', 'https://archiveofourown.org/works/53498773', 'https://archiveofourown.org/works/53485774', 'https://archiveofourown.org/works/53410531', 'https://archiveofourown.org/works/53358775', 'https://archiveofourown.org/works/53275300', 'https://archiveofourown.org/works/53228317', 'https://archiveofourown.org/works/53141194', 'https://archiveofourown.org/works/53209828', 'https://archiveofourown.org/works/53208346', 'https://archiveofourown.org/works/53108299']


In [158]:
fanworks_list4 = remove_chapter_links_test(work_links_pared4)

In [159]:
print(fanworks_list4)

['https://archiveofourown.org/works/54270886', 'https://archiveofourown.org/works/54199738', 'https://archiveofourown.org/works/54066016', 'https://archiveofourown.org/works/54070948', 'https://archiveofourown.org/works/53756806', 'https://archiveofourown.org/works/53629054', 'https://archiveofourown.org/works/53636989', 'https://archiveofourown.org/works/53557639', 'https://archiveofourown.org/works/53531353', 'https://archiveofourown.org/works/53501956', 'https://archiveofourown.org/works/53498773', 'https://archiveofourown.org/works/53485774', 'https://archiveofourown.org/works/53410531', 'https://archiveofourown.org/works/53358775', 'https://archiveofourown.org/works/53275300', 'https://archiveofourown.org/works/53228317', 'https://archiveofourown.org/works/53141194', 'https://archiveofourown.org/works/53209828', 'https://archiveofourown.org/works/53208346', 'https://archiveofourown.org/works/53108299']


In [160]:
print(len(fanworks_list4))

20


In [161]:
all_fanwork_links4.extend(fanworks_list4)

In [162]:
print(all_fanwork_links4)

['https://archiveofourown.org/works/54270886', 'https://archiveofourown.org/works/54199738', 'https://archiveofourown.org/works/54066016', 'https://archiveofourown.org/works/54070948', 'https://archiveofourown.org/works/53756806', 'https://archiveofourown.org/works/53629054', 'https://archiveofourown.org/works/53636989', 'https://archiveofourown.org/works/53557639', 'https://archiveofourown.org/works/53531353', 'https://archiveofourown.org/works/53501956', 'https://archiveofourown.org/works/53498773', 'https://archiveofourown.org/works/53485774', 'https://archiveofourown.org/works/53410531', 'https://archiveofourown.org/works/53358775', 'https://archiveofourown.org/works/53275300', 'https://archiveofourown.org/works/53228317', 'https://archiveofourown.org/works/53141194', 'https://archiveofourown.org/works/53209828', 'https://archiveofourown.org/works/53208346', 'https://archiveofourown.org/works/53108299', 'https://archiveofourown.org/works/54270886', 'https://archiveofourown.org/work